In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
import os

In [ ]:
warnings.filterwarnings("ignore")
os.makedirs("output/charts", exist_ok=True)

In [ ]:
BRAND   = "#E23744"   # Zomato red
ACCENT  = "#FC8019"   # orange
NEUTRAL = "#2D2D2D"
SOFT    = "#F5F5F5"
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "axes.edgecolor": "#DDDDDD", "grid.color": "#EEEEEE"})

Load Data:

In [ ]:
df = pd.read_csv("../enhanced_zomato_dataset_clean.csv")

In [ ]:
#Clean city trailing spaces if any

In [ ]:
df["City"] = df["City"].str.strip()

In [ ]:
#Keep only major cities (drop rows with very few records)

In [ ]:
major_cities = df["City"].value_counts()
major_cities = major_cities[major_cities >= 500].index
df = df[df["City"].isin(major_cities)].copy()

In [ ]:
#Top 12 cuisines by item count for cleaner visuals

In [ ]:
top_cuisines = df["Cuisine"].value_counts().head(12).index
df_top = df[df["Cuisine"].isin(top_cuisines)].copy()

In [ ]:
print(f"Dataset: {df.shape[0]:,} rows | {df['City'].nunique()} cities | {df['Cuisine'].nunique()} cuisines")
print(f"Restaurants: {df['Restaurant_Name'].nunique()} | Top-cuisine subset: {df_top.shape[0]:,} rows\n")

Section 1- Restaurant level dataset

In [ ]:
rest = (
    df.groupby(["Restaurant_Name", "City", "Cuisine"])
    .agg(
        Dining_Rating      = ("Dining_Rating",        "first"),
        Delivery_Rating    = ("Delivery_Rating",       "first"),
        Avg_Rating         = ("Average_Rating",        "mean"),
        Avg_Price          = ("Prices",                "mean"),
        Total_Items        = ("Item_Name",             "count"),
        Bestseller_Items   = ("Is_Bestseller",         "sum"),
        Total_Votes        = ("Total_Votes",           "first"),
        Popularity         = ("Restaurant_Popularity", "first"),
        Is_Highly_Rated    = ("Is_Highly_Rated",       "first"),
        Is_Expensive       = ("Is_Expensive",          "first"),
    )
    .reset_index()
)
rest["Bestseller_Rate"] = rest["Bestseller_Items"] / rest["Total_Items"]
 

In [ ]:
df["Is_BS_Raw"] = df["Best_Seller"].isin(["BESTSELLER", "MUST TRY"]).astype(int)
print(f"Restaurant-level dataset: {rest.shape[0]} restaurants\n")

SECTION 2 - CITY × CUISINE SATURATION MATRIX

In [ ]:
sat = (
    df_top.groupby(["City", "Cuisine"])["Restaurant_Name"]
    .nunique()
    .reset_index(name="Restaurant_Count")
)

In [ ]:
city_total = rest.groupby("City")["Restaurant_Name"].nunique().reset_index(name="City_Total")
sat = sat.merge(city_total, on="City")
sat["Saturation_Index"] = (sat["Restaurant_Count"] / sat["City_Total"] * 100).round(1)

In [ ]:
hhi = (
    sat.groupby("City")
    .apply(lambda x: (x["Saturation_Index"] / 100 ** 2 * 100).sum())
    .reset_index(name="Concentration_Score")
)
print("\nCity Concentration Score (lower = more diverse cuisine landscape):")
print(hhi.sort_values("Concentration_Score").to_string(index=False))

In [ ]:
sat_pivot = sat.pivot(index="Cuisine", columns="City", values="Saturation_Index").fillna(0)
 
fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    sat_pivot, annot=True, fmt=".0f", cmap="YlOrRd",
    linewidths=0.4, linecolor="#eeeeee",
    annot_kws={"size": 8}, ax=ax, cbar_kws={"label": "% of city restaurants"}
)
ax.set_title("Cuisine Saturation by City  (% of city restaurants per cuisine)",
             fontsize=13, fontweight="bold", pad=12, color=NEUTRAL)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=30, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig("output/charts/01_saturation_heatmap.png", dpi=150)
plt.close()

SECTION 3 - MARKET GAPS (underserved cuisine × city combos)

In [ ]:
cuisine_national_rating = (
    df_top.groupby("Cuisine")["Average_Rating"].mean().reset_index(name="National_Avg_Rating")
)
gaps = sat.merge(cuisine_national_rating, on="Cuisine")
gaps["Gap_Score"] = (gaps["National_Avg_Rating"] * (1 - gaps["Saturation_Index"] / 100)).round(3)
 

In [ ]:
top_gaps = gaps.sort_values("Gap_Score", ascending=False).head(15)
print("\nTop Market Gaps (high-rated cuisine, low city presence):")
print(top_gaps[["City", "Cuisine", "Restaurant_Count", "Saturation_Index", "National_Avg_Rating", "Gap_Score"]]
      .to_string(index=False))
 
fig, ax = plt.subplots(figsize=(11, 6))
colors = [BRAND if i < 5 else ACCENT for i in range(len(top_gaps))]
bars = ax.barh(
    top_gaps["City"] + " — " + top_gaps["Cuisine"],
    top_gaps["Gap_Score"], color=colors
)
ax.set_title("Top 15 Market Opportunity Gaps\n(High national rating × Low city presence)",
             fontsize=12, fontweight="bold", color=NEUTRAL)
ax.set_xlabel("Gap Score")
ax.invert_yaxis()
for bar, val in zip(bars, top_gaps["Gap_Score"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("output/charts/02_market_gaps.png", dpi=150)
plt.close()

SECTION 4 - HYPE VS QUALITY (Popularity vs Rating)

In [ ]:
#Normalising both to 0-1
rest["Pop_Norm"]    = (rest["Popularity"] - rest["Popularity"].min()) / (rest["Popularity"].max() - rest["Popularity"].min())
rest["Rating_Norm"] = (rest["Avg_Rating"] - rest["Avg_Rating"].min()) / (rest["Avg_Rating"].max() - rest["Avg_Rating"].min())
rest["Hype_Gap"]    = (rest["Pop_Norm"] - rest["Rating_Norm"]).round(3) 

In [ ]:
def quadrant(row):
    p, r = row["Pop_Norm"], row["Rating_Norm"]
    if p >= 0.5 and r >= 0.5: return "Star (Popular & Rated)"
    if p >= 0.5 and r <  0.5: return "Overhyped"
    if p <  0.5 and r >= 0.5: return "Hidden Gem"
    return "Underperformer"
 
rest["Quadrant"] = rest.apply(quadrant, axis=1)
print("\nQuadrant distribution:")
print(rest["Quadrant"].value_counts().to_string())
 
q_palette = {
    "Star (Popular & Rated)": "#2ecc71",
    "Overhyped":              BRAND,
    "Hidden Gem":             "#3498db",
    "Underperformer":         "#95a5a6",
}
 
fig, ax = plt.subplots(figsize=(10, 7))
for quad, grp in rest.groupby("Quadrant"):
    ax.scatter(grp["Pop_Norm"], grp["Rating_Norm"],
               label=quad, alpha=0.55, s=25, color=q_palette[quad])
 
ax.axhline(0.5, color="#AAAAAA", lw=1, ls="--")
ax.axvline(0.5, color="#AAAAAA", lw=1, ls="--")
ax.text(0.75, 0.92, "Stars", fontsize=9, color="#2ecc71", transform=ax.transAxes)
ax.text(0.75, 0.08, "Overhyped", fontsize=9, color=BRAND, transform=ax.transAxes)
ax.text(0.05, 0.92, "Hidden Gems", fontsize=9, color="#3498db", transform=ax.transAxes)
ax.text(0.05, 0.08, "Underperformers", fontsize=9, color="#95a5a6", transform=ax.transAxes)
ax.set_xlabel("Popularity (normalised)", fontsize=10)
ax.set_ylabel("Avg Rating (normalised)", fontsize=10)
ax.set_title("Hype vs Quality — Restaurant Quadrant Map", fontsize=12, fontweight="bold", color=NEUTRAL)
ax.legend(loc="upper left", fontsize=8, framealpha=0.6)
plt.tight_layout()
plt.savefig("output/charts/03_hype_vs_quality.png", dpi=150)
plt.close() 

In [ ]:
#Top overhyped $ hidden gems
print("\nTop Overhyped Restaurants (popular but low-rated):")
print(rest[rest["Quadrant"] == "Overhyped"]
      .sort_values("Hype_Gap", ascending=False)
      [["Restaurant_Name", "City", "Cuisine", "Popularity", "Avg_Rating", "Hype_Gap"]]
      .head(10).to_string(index=False))
 
print("\nTop Hidden Gems (high-rated but under-the-radar):")
print(rest[rest["Quadrant"] == "Hidden Gem"]
      .sort_values("Hype_Gap")
      [["Restaurant_Name", "City", "Cuisine", "Popularity", "Avg_Rating", "Hype_Gap"]]
      .head(10).to_string(index=False))

SECTION 5 - BESTSELLER RATE BY CUISINE & CITY

In [ ]:
#Item-level bestseller rate per cuisine

In [ ]:
df_top = df_top.copy()
df_top["Is_BS_Raw"] = df_top["Best_Seller"].isin(["BESTSELLER", "MUST TRY"]).astype(int)
 
bs_cuisine = (
    df_top.groupby("Cuisine")
    .agg(Total_Items=("Item_Name", "count"), BS_Items=("Is_BS_Raw", "sum"))
    .reset_index()
)
bs_cuisine["BS_Rate"] = (bs_cuisine["BS_Items"] / bs_cuisine["Total_Items"] * 100).round(1)
bs_cuisine = bs_cuisine.sort_values("BS_Rate", ascending=False)
 
print("\nBestseller Rate by Cuisine:")
print(bs_cuisine[["Cuisine", "Total_Items", "BS_Items", "BS_Rate"]].to_string(index=False))
 
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(bs_cuisine["Cuisine"], bs_cuisine["BS_Rate"],
               color=[BRAND if x >= bs_cuisine["BS_Rate"].median() else ACCENT
                      for x in bs_cuisine["BS_Rate"]])
ax.set_xlabel("Bestseller Rate (%)")
ax.set_title("Bestseller Conversion Rate by Cuisine\n(% of items tagged BESTSELLER)",
             fontsize=12, fontweight="bold", color=NEUTRAL)
ax.axvline(bs_cuisine["BS_Rate"].median(), color="#555", ls="--", lw=1.2, label="Median")
ax.invert_yaxis()
ax.legend(fontsize=9)
for bar, val in zip(bars, bs_cuisine["BS_Rate"]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{val}%", va="center", fontsize=8.5)
plt.tight_layout()
plt.savefig("output/charts/04_bestseller_rate.png", dpi=150)
plt.close()

SECTION 6 - PRICE-VALUE QUADRANT (Price vs Rating by Cuisine)

In [ ]:
pv = (
    df_top.groupby("Cuisine")
    .agg(Avg_Price=("Prices", "mean"), Avg_Rating=("Average_Rating", "mean"),
         Count=("Restaurant_Name", "nunique"))
    .reset_index()
)
 
price_med  = pv["Avg_Price"].median()
rating_med = pv["Avg_Rating"].median()
 
def pv_quadrant(row):
    if row["Avg_Price"] <= price_med and row["Avg_Rating"] >= rating_med: return "Best Value"
    if row["Avg_Price"] >  price_med and row["Avg_Rating"] >= rating_med: return "Premium Quality"
    if row["Avg_Price"] <= price_med and row["Avg_Rating"] <  rating_med: return "Budget / Risky"
    return "Overpriced"
 
pv["PV_Quadrant"] = pv.apply(pv_quadrant, axis=1)
print("\nPrice-Value Quadrant by Cuisine:")
print(pv[["Cuisine", "Avg_Price", "Avg_Rating", "PV_Quadrant"]].sort_values("PV_Quadrant").to_string(index=False))
 
pv_palette = {
    "Best Value":      "#2ecc71",
    "Premium Quality": "#3498db",
    "Budget / Risky":  "#e67e22",
    "Overpriced":      BRAND,
}
 
fig, ax = plt.subplots(figsize=(10, 7))
for quad, grp in pv.groupby("PV_Quadrant"):
    ax.scatter(grp["Avg_Price"], grp["Avg_Rating"],
               s=grp["Count"] * 25, alpha=0.8,
               color=pv_palette[quad], label=quad, edgecolors="white", linewidth=0.5)
    for _, row in grp.iterrows():
        ax.annotate(row["Cuisine"], (row["Avg_Price"], row["Avg_Rating"]),
                    fontsize=7.5, ha="center", va="bottom",
                    xytext=(0, 5), textcoords="offset points")
 
ax.axhline(rating_med, color="#AAAAAA", ls="--", lw=1)
ax.axvline(price_med,  color="#AAAAAA", ls="--", lw=1)
ax.set_xlabel("Avg Item Price (₹)", fontsize=10)
ax.set_ylabel("Avg Rating", fontsize=10)
ax.set_title("Price-Value Positioning by Cuisine\n(bubble size = no. of restaurants)",
             fontsize=12, fontweight="bold", color=NEUTRAL)
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.savefig("output/charts/05_price_value_quadrant.png", dpi=150)
plt.close()

SECTION 7 - CITY-LEVEL OVERVIEW

In [ ]:
city_summary = (
    df.groupby("City")
    .agg(
        Restaurants       = ("Restaurant_Name",    "nunique"),
        Cuisines          = ("Cuisine",            "nunique"),
        Avg_Rating        = ("Average_Rating",     "mean"),
        Avg_Price         = ("Prices",             "mean"),
        Bestseller_Rate   = ("Is_Bestseller",      "mean"),
        Highly_Rated_Pct  = ("Is_Highly_Rated",    "mean"),
    )
    .reset_index()
    .sort_values("Restaurants", ascending=False)
)
city_summary["Avg_Rating"]       = city_summary["Avg_Rating"].round(2)
city_summary["Avg_Price"]        = city_summary["Avg_Price"].round(0)
city_summary["Bestseller_Rate"]  = (city_summary["Bestseller_Rate"] * 100).round(1)
city_summary["Highly_Rated_Pct"] = (city_summary["Highly_Rated_Pct"] * 100).round(1)
 
print(city_summary.to_string(index=False))
 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cs = city_summary.sort_values("Avg_Rating", ascending=True)
 
axes[0].barh(cs["City"], cs["Avg_Rating"], color=BRAND, alpha=0.85)
axes[0].set_xlabel("Avg Rating")
axes[0].set_title("Average Rating by City", fontweight="bold", color=NEUTRAL)
for i, (val, city) in enumerate(zip(cs["Avg_Rating"], cs["City"])):
    axes[0].text(val - 0.05, i, f"{val:.2f}", va="center", ha="right",
                 color="white", fontsize=8.5, fontweight="bold")
 
cs2 = city_summary.sort_values("Avg_Price", ascending=True)
axes[1].barh(cs2["City"], cs2["Avg_Price"], color=ACCENT, alpha=0.85)
axes[1].set_xlabel("Avg Item Price (₹)")
axes[1].set_title("Average Item Price by City", fontweight="bold", color=NEUTRAL)
for i, val in enumerate(cs2["Avg_Price"]):
    axes[1].text(val - 3, i, f"₹{val:.0f}", va="center", ha="right",
                 color="white", fontsize=8.5, fontweight="bold")
 
plt.suptitle("City-Level Competitive Overview", fontsize=13, fontweight="bold",
             color=NEUTRAL, y=1.02)
plt.tight_layout()
plt.savefig("output/charts/06_city_overview.png", dpi=150, bbox_inches="tight")
plt.close()

SECTION 8 - EXPORTs FOR POWER BI

In [ ]:
# Table 1: Restaurant-level master
rest_export = rest.copy()
rest_export.to_csv("output/pb_restaurants.csv", index=False)
# Table 2: City-cuisine saturation
sat_export = sat.copy()
sat_export.to_csv("output/pb_saturation.csv", index=False)
 
# Table 3: City summary
city_summary.to_csv("output/pb_city_summary.csv", index=False)
 
# Table 4: Cuisine price-value
pv.to_csv("output/pb_cuisine_pv.csv", index=False)
 
# Table 5: Market gaps
gaps.to_csv("output/pb_market_gaps.csv", index=False)
 
# Table 6: Bestseller rates by cuisine
bs_cuisine.to_csv("output/pb_bestseller.csv", index=False)

SECTION 9 - KEY FINDINGS SUMMARY

In [ ]:
top_gap     = top_gaps.iloc[0]
top_hyped   = rest[rest["Quadrant"] == "Overhyped"].sort_values("Hype_Gap", ascending=False).iloc[0]
top_gem     = rest[rest["Quadrant"] == "Hidden Gem"].sort_values("Hype_Gap").iloc[0]
top_bv      = pv[pv["PV_Quadrant"] == "Best Value"].sort_values("Avg_Rating", ascending=False).iloc[0]
most_div    = hhi.sort_values("Concentration_Score").iloc[0]
least_div   = hhi.sort_values("Concentration_Score", ascending=False).iloc[0]
 
print(f"""
1. SATURATION
   Most cuisine-diverse city : {most_div['City']} (HHI score: {most_div['Concentration_Score']:.1f})
   Most concentrated city    : {least_div['City']} (HHI score: {least_div['Concentration_Score']:.1f})
 
2. MARKET GAPS (entry opportunities)
   Top gap: {top_gap['Cuisine']} in {top_gap['City']}
   → Only {top_gap['Restaurant_Count']} restaurant(s), {top_gap['Saturation_Index']}% city share,
     national avg rating {top_gap['National_Avg_Rating']:.2f}
 
3. HYPE vs QUALITY
   Most overhyped : {top_hyped['Restaurant_Name']} ({top_hyped['City']}) — Hype Gap: {top_hyped['Hype_Gap']:.2f}
   Top hidden gem : {top_gem['Restaurant_Name']} ({top_gem['City']}) — Hype Gap: {top_gem['Hype_Gap']:.2f}
 
4. PRICE-VALUE WINNERS
   Best-value cuisine: {top_bv['Cuisine']}
   → Avg ₹{top_bv['Avg_Price']:.0f}, Avg Rating {top_bv['Avg_Rating']:.2f}
 
5. BESTSELLER CONVERSION
   Highest BS rate : {bs_cuisine.iloc[0]['Cuisine']} ({bs_cuisine.iloc[0]['BS_Rate']}%)
   Lowest BS rate  : {bs_cuisine.iloc[-1]['Cuisine']} ({bs_cuisine.iloc[-1]['BS_Rate']}%)
""")